In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": { "parameterName": "LakehouseName", "defaultValue": "AzureCostAnalyzer_LH" },
    "id": { "parameterName": "LakehouseId", "defaultValue": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx" },
    "workspaceId": { "parameterName": "WorkspaceId", "defaultValue": "yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy" }
  }
}

# 03 · Gold · Focus Monthly

Builds **`gold_cost_focus_monthly`** — an enriched monthly cost fact from `focus_partitioned`, exposing FOCUS + Azure-extension columns that `gold_cost_summary_monthly` does not carry.

## New dimensions vs `gold_cost_summary_monthly`
- `ResourceGroupName` (`x_ResourceGroupName`)
- `RegionName` (was only in daily)
- `ChargeCategory` (Usage / Purchase / Tax / Adjustment / Credit)
- `PricingCategory` (Standard / Reservation / Spot / SavingsPlan)
- `CommitmentDiscountStatus` (Used / Unused / N/A) — splits reservation usage from idle reserved capacity
- `CostCenter` (`x_CostCenter`)
- `ContractedCost`
- `EffectiveCostUSD` (`x_EffectiveCostInUsd`)
- `ResourceCount` (DISTINCT ResourceId)

## Savings logic (hybrid baseline)
`SavingsAmount = SavingsBaseline − EffectiveCost`, where the baseline is FOCUS-correct:
- **Non-commitment** (`PricingCategory != 'Committed'`) → `ListCost` (discount vs public list).
- **Commitment** (`PricingCategory == 'Committed'`) → `ContractedCost` (savings vs your
  negotiated on-demand rate; the commitment discount itself is the value being measured).

Reason: amortized reservation rows carry `ListCost = 0`, so the naive `ListCost − EffectiveCost`
yields a false negative. `ContractedCost` is populated for commitments and is the on-demand-equivalent
baseline. Idle reservation rows (`CommitmentDiscountStatus = 'Unused'`) have `ContractedCost = 0`,
so their `EffectiveCost` surfaces correctly as waste. `SavingsBaseline` is persisted for transparency.

## Filter
Includes ALL ChargeCategory values (no exclusions) for flexible reporting.

## Dependencies
- `focus_partitioned` (silver) with FOCUS + `x_*` extensions

In [ ]:
from pyspark.sql import functions as F

In [ ]:
# Parameters
SOURCE_TABLE = "focus_partitioned"
OUTPUT_TABLE = "gold_cost_focus_monthly"

## Step 1 · Aggregate from `focus_partitioned`

Group by YearMonth + enriched dimensions, summing costs and counting resources.

In [ ]:
# Partition pruning: read last 12 months only
from datetime import date
from dateutil.relativedelta import relativedelta

lookback_date = date.today() - relativedelta(months=12)

# Hybrid savings baseline (FOCUS-correct):
#   - Non-commitment usage  -> ListCost      (discount vs public list price)
#   - Commitment usage       -> ContractedCost (savings vs negotiated on-demand rate,
#                                               excludes the commitment discount itself)
# Rationale: amortized reservation rows carry ListCost = 0, so List - Effective yields a
# false negative. ContractedUnitPrice is populated for commitments and represents the
# on-demand-equivalent at your negotiated rate. Unused reservation rows have
# ContractedCost = 0, so their full EffectiveCost surfaces correctly as waste.
savings_baseline = F.when(
    F.col("PricingCategory") == "Committed", F.col("ContractedCost")
).otherwise(F.col("ListCost"))

df = (
    spark.table(SOURCE_TABLE)
         .filter(
             (F.col("Year") > lookback_date.year) | 
             ((F.col("Year") == lookback_date.year) & (F.col("Month") >= lookback_date.month))
         )
         .withColumn("YearMonth", F.date_format(F.col("ChargePeriodStart"), "yyyy-MM"))
         .groupBy(
             "YearMonth",
             "SubAccountName",
             F.col("x_ResourceGroupName").alias("ResourceGroupName"),
             "ServiceCategory",
             "ServiceName",
             "RegionName",
             "ChargeCategory",
             "PricingCategory",
             F.coalesce(F.col("CommitmentDiscountStatus"), F.lit("N/A")).alias("CommitmentDiscountStatus"),
             F.col("x_CostCenter").alias("CostCenter"),
         )
         .agg(
             F.sum("EffectiveCost").alias("EffectiveCost"),
             F.sum("BilledCost").alias("BilledCost"),
             F.sum("ListCost").alias("ListCost"),
             F.sum("ContractedCost").alias("ContractedCost"),
             F.sum(savings_baseline).alias("SavingsBaseline"),
             F.sum(savings_baseline - F.col("EffectiveCost")).alias("SavingsAmount"),
             F.sum("x_EffectiveCostInUsd").alias("EffectiveCostUSD"),
             F.countDistinct("ResourceId").alias("ResourceCount"),
         )
)

print(f"Aggregated {df.count():,} rows from {SOURCE_TABLE}")
df.show(10, truncate=False)

## Step 2 · Write to `gold_cost_focus_monthly`

Mode: overwrite (full refresh). No partition (monthly grain is small enough).

In [ ]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
print(f"✓ Wrote {df.count():,} rows to {OUTPUT_TABLE}")

## Step 3 · Validate output

Check schema, row count, cost totals, and dimension cardinalities.

In [ ]:
# Schema
spark.table(OUTPUT_TABLE).printSchema()

# Aggregate stats
stats = (
    spark.table(OUTPUT_TABLE)
         .select(
             F.count("*").alias("RowCount"),
             F.countDistinct("YearMonth").alias("Months"),
             F.countDistinct("SubAccountName").alias("Subscriptions"),
             F.countDistinct("ResourceGroupName").alias("ResourceGroups"),
             F.countDistinct("ServiceName").alias("Services"),
             F.countDistinct("ChargeCategory").alias("ChargeCategories"),
             F.sum("EffectiveCost").alias("TotalEffectiveCost"),
             F.sum("SavingsAmount").alias("TotalSavings"),
             F.sum("ResourceCount").alias("TotalResourceCount"),
         )
         .first()
)

print("="*70)
print(f"Rows                 : {stats.RowCount:,}")
print(f"Months               : {stats.Months}")
print(f"Subscriptions        : {stats.Subscriptions}")
print(f"Resource Groups      : {stats.ResourceGroups}")
print(f"Services             : {stats.Services}")
print(f"Charge Categories    : {stats.ChargeCategories}")
print(f"Total Effective Cost : ${stats.TotalEffectiveCost:,.2f}")
print(f"Total Savings        : ${stats.TotalSavings:,.2f}")
print(f"Total Resource Count : {stats.TotalResourceCount:,}")
print("="*70)

# Sample rows (top months by cost)
print("\nTop 5 YearMonth by Effective Cost:")
(spark.table(OUTPUT_TABLE)
      .groupBy("YearMonth")
      .agg(F.sum("EffectiveCost").alias("MonthCost"))
      .orderBy(F.desc("MonthCost"))
      .show(5, truncate=False))